# Experiment 1: SSM Induction Score on Zamba2-1.2B

**Paper:** Mechanistic Interpretability of Hybrid SSM-Attention Models  
**RQ1:** Do SSM layers in hybrid models develop induction-like behavior analogous to attention induction heads?

## Method: Logit Lens SSMI

For each layer L, apply `ln_final + unembed` to `hook_out[L]` to get "what would this layer predict if it were the final layer?"  
**SSMI(L)** = mean logprob of the correct induction token at induction positions.

Requires only **1 `run_with_cache` call per sequence** (not 38 patched runs — those are too slow on T4).  
Only `blocks.*.hook_out` is cached (38 tensors, not ~450 submodule hooks).

## Architecture
Zamba2-1.2B: 38 layers — 32 × `"mamba"` (pure SSM) + 6 × `"hybrid"` (SSM + shared attention)

In [ ]:
# ── 1. Install ───────────────────────────────────────────────────────────────
!pip install -q git+https://github.com/TransformerLensOrg/TransformerLens.git@dev
!pip install -q transformers>=4.47.0 einops jaxtyping

In [ ]:
# ── 2. Imports ───────────────────────────────────────────────────────────────
import gc, random
from typing import List, Tuple
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import torch
from transformer_lens.model_bridge.bridge import TransformerBridge

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16 if torch.cuda.is_available() else torch.float32
MODEL  = "Zyphra/Zamba2-1.2B"
print(f"Device: {DEVICE} | dtype: {DTYPE}")

In [ ]:
# ── 3. Load model ────────────────────────────────────────────────────────────
bridge = TransformerBridge.boot_transformers(MODEL, device=DEVICE, dtype=DTYPE)
n_layers = len(bridge.blocks)
lbt = list(getattr(bridge.cfg, "layers_block_type", []))
print(f"Layers: {n_layers} | Mamba: {lbt.count('mamba')} | Hybrid: {lbt.count('hybrid')}")

In [ ]:
# ── 4. Build induction sequences ─────────────────────────────────────────────
# ABC→BCD: [prefix A] [SEP] [prefix A]  — second copy should predict A[i+1]
PREFIX_LEN = 30   # reduced for T4 speed
N_SEQS     = 10   # pilot run (enough to see the per-layer pattern)
VOCAB_SIZE = bridge.cfg.d_vocab
random.seed(42); torch.manual_seed(42)

def make_induction_seq(prefix_len):
    prefix = torch.randint(100, VOCAB_SIZE - 100, (prefix_len,))
    sep    = torch.tensor([1])
    clean  = torch.cat([prefix, sep, prefix]).unsqueeze(0)
    return clean

seqs = [make_induction_seq(PREFIX_LEN) for _ in range(N_SEQS)]
INDUCTION_POSITIONS = list(range(PREFIX_LEN + 1, 2 * PREFIX_LEN))
print(f"Seq shape: {seqs[0].shape} | Induction positions: {len(INDUCTION_POSITIONS)}")

In [ ]:
# ── 5. Logit-lens SSMI (fast: 1 forward pass per sequence) ───────────────────
# GPU warmup — triggers Mamba2 CUDA JIT kernel compilation (~5 min first time)
print("GPU warmup (first Mamba2 CUDA call — takes up to 5 min to JIT compile kernels)...")
with torch.no_grad():
    _ = bridge(torch.tensor([[1, 2, 3, 4, 5]]).to(DEVICE))
if torch.cuda.is_available(): torch.cuda.synchronize()
print("Warmup done. All subsequent forward passes are fast.")

# Unembedding matrix and final norm for logit projection
try:
    W_U = bridge.unembed.original_module.weight.detach().float()   # [vocab, d_model]
    ln_final_mod = bridge.ln_final.original_module
except Exception:
    W_U = bridge.original_model.lm_head.weight.detach().float()
    ln_final_mod = bridge.original_model.model.final_layernorm

@torch.no_grad()
def ssmi_logit_lens(bridge, seqs, induction_positions):
    """Logit-lens SSMI: 1 fwd pass per seq, only blocks.*.hook_out cached."""
    n_layers = len(bridge.blocks)
    layer_scores = np.zeros(n_layers)
    # Only cache block-level hook_out — 38 hooks instead of ~450
    hook_filter = lambda name: name.endswith("hook_out") and name.startswith("blocks.")

    for seq_idx, clean in enumerate(seqs):
        print(f"  Sequence {seq_idx+1}/{len(seqs)}...", flush=True)
        clean = clean.to(DEVICE)
        _, cache = bridge.run_with_cache(clean, names_filter=hook_filter)

        for layer_idx in range(n_layers):
            key = f"blocks.{layer_idx}.hook_out"
            if key not in cache: continue
            residual = cache[key].float()               # [1, seq_len, d_model]
            normed   = ln_final_mod(residual)            # [1, seq_len, d_model]
            logits   = normed @ W_U.T                    # [1, seq_len, vocab]
            lp       = torch.log_softmax(logits[0], dim=-1)
            scores   = [lp[pos, clean[0, pos+1].item()].item()
                        for pos in induction_positions if pos+1 < clean.shape[1]]
            if scores: layer_scores[layer_idx] += float(np.mean(scores))

        del cache
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    layer_scores /= len(seqs)
    return layer_scores

print("Computing logit-lens SSMI per layer...")
ssmi_scores = ssmi_logit_lens(bridge, seqs, INDUCTION_POSITIONS)
print(f"\nDone. Max SSMI: {ssmi_scores.max():.3f} at layer {ssmi_scores.argmax()} ({lbt[ssmi_scores.argmax()]})") 

In [ ]:
# ── 6. Plot ───────────────────────────────────────────────────────────────────
colors = ["#e05c5c" if t == "hybrid" else "#5c8de0" for t in lbt]
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(np.arange(n_layers), ssmi_scores, color=colors, width=0.8, alpha=0.85)
for pos in [i for i, t in enumerate(lbt) if t == "hybrid"]:
    ax.axvline(x=pos, color="#cc3333", linewidth=0.8, linestyle="--", alpha=0.5)
ax.legend(handles=[
    Patch(facecolor="#5c8de0", label="Mamba-2 (SSM) layer"),
    Patch(facecolor="#e05c5c", label="Hybrid layer (SSM + shared attention)"),
], loc="upper left", fontsize=10)
ax.set_xlabel("Layer index", fontsize=12)
ax.set_ylabel("SSMI — log-prob of induction token (logit lens)", fontsize=11)
ax.set_title("SSM Induction Score per layer — Zamba2-1.2B (logit lens)", fontsize=13)
ax.axhline(y=0, color="black", linewidth=0.8)
ax.set_xlim(-0.5, n_layers - 0.5)
plt.tight_layout()
plt.savefig("ssmi_zamba2_1.2b.pdf", bbox_inches="tight", dpi=150)
plt.savefig("ssmi_zamba2_1.2b.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved: ssmi_zamba2_1.2b.pdf + .png")

In [ ]:
# ── 7. Summary ────────────────────────────────────────────────────────────────
mamba_s  = ssmi_scores[[i for i, t in enumerate(lbt) if t == "mamba"]]
hybrid_s = ssmi_scores[[i for i, t in enumerate(lbt) if t == "hybrid"]]
print(f"Mamba  layers — mean: {mamba_s.mean():.4f}, max: {mamba_s.max():.4f}")
print(f"Hybrid layers — mean: {hybrid_s.mean():.4f}, max: {hybrid_s.max():.4f}")
print("\nTop-5 layers:")
for rank, idx in enumerate(np.argsort(ssmi_scores)[::-1][:5]):
    print(f"  {rank+1}. Layer {idx:2d} ({lbt[idx]:6s}) SSMI={ssmi_scores[idx]:.4f}")

In [ ]:
# ── 8. Save results ───────────────────────────────────────────────────────────
import json
with open("ssmi_zamba2_results.json", "w") as f:
    json.dump({
        "model": MODEL, "method": "logit_lens",
        "n_sequences": N_SEQS, "prefix_len": PREFIX_LEN,
        "layer_types": lbt, "ssmi_scores": ssmi_scores.tolist(),
        "summary": {
            "max_layer": int(ssmi_scores.argmax()), "max_layer_type": lbt[ssmi_scores.argmax()],
            "mamba_mean": float(mamba_s.mean()), "hybrid_mean": float(hybrid_s.mean()),
        }
    }, f, indent=2)
print("Saved: ssmi_zamba2_results.json")

## Interpretation

**What to look for in the plot:**
- If hybrid layers (red) have highest SSMI → shared attention seeds the induction pattern, Mamba amplifies
- If Mamba layers (blue) have highest SSMI and it rises monotonically → SSM builds induction without attention
- If SSMI peaks just *after* each hybrid layer → attention seeds, subsequent SSM amplifies

All three are publishable findings. The logit lens shows *where* the pattern builds, not *what causes it* (for causality, use activation patching — but that requires a faster GPU like A100).